# ЛР-02: Школьное питание

## Student notebook: civil 02

Этот notebook предназначен для самостоятельного решения.

Готового оптимального плана и заполненного решателя здесь нет.

## 1. Зачем нужен этот кейс

Три комбината развозят питание по школам, но общий запас больше общего спроса.

После этого notebook-а студент должен уметь:

1. видеть небаланс сразу по суммам запасов и спроса;
2. добавлять фиктивного потребителя и объяснять его смысл;
3. аккуратно расширять матрицу затрат;
4. отделять реальные маршруты от фиктивного остатка.

## 2. Исходные данные

Тип задачи: **открытая, запасов больше, чем спроса**.

### Запасы

| Поставщик | Объём |
| --- | --- |
| Комбинат A | 40 |
| Комбинат B | 35 |
| Комбинат C | 30 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Школа 1 | 20 |
| Школа 2 | 25 |
| Школа 3 | 30 |
| Школа 4 | 15 |

### Матрица затрат

| Откуда / Куда | Школа 1 | Школа 2 | Школа 3 | Школа 4 |
| --- | --- | --- | --- | --- |
| Комбинат A | 4 | 6 | 8 | 7 |
| Комбинат B | 5 | 4 | 7 | 6 |
| Комбинат C | 6 | 5 | 4 | 8 |

## 3. Что нужно сделать

Идите тем же маршрутом, что и в теории ЛР-02:

1. Проверьте баланс запасов и спроса.
2. Если задача открытая, добавьте фиктивный узел и поясните его смысл.
3. Запишите математическую модель через переменные $x_{ij}$.
4. Сформируйте `c`, `A_eq`, `b_eq`, `bounds` для `linprog`.
5. Проверьте, что `A_eq x = b_eq` действительно задаёт строки и столбцы.
6. После собственной попытки решите задачу через Python.
7. Кратко объясните реальные и фиктивные маршруты.


In [6]:
def balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
):
    """Готовит закрытую транспортную модель через фиктивный узел."""

    BALANCE_TOLERANCE = 1e-9
    DUMMY_CONSUMER_NAME = "Фиктивный потребитель"
    DUMMY_SUPPLIER_NAME = "Фиктивный поставщик"

    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    # Добавляем фиктивного потребителя, если запасов больше
    if balance_difference > BALANCE_TOLERANCE:
        # Добавляем фиктивного потребителя
        balanced_demands = np.append(balanced_demands, balance_difference)
        balanced_consumer_names.append(DUMMY_CONSUMER_NAME)

        # Добавляем столбец с нулевыми затратами для фиктивного потребителя
        dummy_column = np.full((balanced_costs.shape[0], 1), dummy_cost)
        balanced_costs = np.column_stack([balanced_costs, dummy_column])

        balance_note = f"Задача открытая: запасов больше на {balance_difference:.1f}. Добавлен фиктивный потребитель."

    # Добавляем фиктивного поставщика, если спроса больше
    elif balance_difference < -BALANCE_TOLERANCE:
        # Добавляем фиктивного поставщика
        balanced_supplies = np.append(balanced_supplies, -balance_difference)
        balanced_supplier_names.append(DUMMY_SUPPLIER_NAME)

        # Добавляем строку с нулевыми затратами для фиктивного поставщика
        dummy_row = np.full((1, balanced_costs.shape[1]), dummy_cost)
        balanced_costs = np.vstack([balanced_costs, dummy_row])

        balance_note = f"Задача открытая: спроса больше на {-balance_difference:.1f}. Добавлен фиктивный поставщик."

    else:
        balance_note = "Задача закрытая: сумма запасов = сумме спроса"

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )

## 4. Шаблон для самостоятельной сборки модели

Ниже намеренно оставлены `TODO`. Это не готовое решение, а guided skeleton:
он показывает правильные имена, порядок шагов и форму LP-модели, но ключевые
строки вы заполняете самостоятельно.


In [2]:
def balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
):
    """Готовит закрытую транспортную модель через фиктивный узел."""

    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    # Добавляем фиктивного потребителя, если запасов больше
    if balance_difference > BALANCE_TOLERANCE:
        # Добавляем фиктивного потребителя
        balanced_demands = np.append(balanced_demands, balance_difference)
        balanced_consumer_names.append(DUMMY_CONSUMER_NAME)

        # Добавляем столбец с нулевыми затратами для фиктивного потребителя
        dummy_column = np.full((balanced_costs.shape[0], 1), dummy_cost)
        balanced_costs = np.column_stack([balanced_costs, dummy_column])

        balance_note = f"Задача открытая: запасов больше на {balance_difference:.1f}. Добавлен фиктивный потребитель."

    # Добавляем фиктивного поставщика, если спроса больше
    elif balance_difference < -BALANCE_TOLERANCE:
        # Добавляем фиктивного поставщика
        balanced_supplies = np.append(balanced_supplies, -balance_difference)
        balanced_supplier_names.append(DUMMY_SUPPLIER_NAME)

        # Добавляем строку с нулевыми затратами для фиктивного поставщика
        dummy_row = np.full((1, balanced_costs.shape[1]), dummy_cost)
        balanced_costs = np.vstack([balanced_costs, dummy_row])

        balance_note = f"Задача открытая: спроса больше на {-balance_difference:.1f}. Добавлен фиктивный поставщик."

    else:
        balance_note = "Задача закрытая: сумма запасов = сумме спроса"

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )

In [3]:
def build_transport_lp(supplies, demands, costs):
    """Собирает c, A_eq, b_eq и bounds для linprog."""

    supplier_count, consumer_count = costs.shape
    variable_count = supplier_count * consumer_count

    # Вектор цели (стоимости перевозок)
    c = costs.flatten()

    # Матрица ограничений A_eq
    A_eq = np.zeros((supplier_count + consumer_count, variable_count))

    # Заполняем строки поставщиков
    for i in range(supplier_count):
        for j in range(consumer_count):
            idx = i * consumer_count + j
            A_eq[i, idx] = 1

    # Заполняем строки потребителей
    for j in range(consumer_count):
        for i in range(supplier_count):
            idx = i * consumer_count + j
            A_eq[supplier_count + j, idx] = 1

    # Вектор правых частей
    b_eq = np.concatenate([supplies, demands])

    # Границы переменных
    bounds = [(0.0, None) for _ in range(variable_count)]

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
        "route_index_hint": "supplier_idx * consumer_count + consumer_idx",
        "variable_count": variable_count,
    }

In [7]:
# Шаг 1: сбалансируйте задачу
(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
    balanced_supplier_names,
    balanced_consumer_names,
    balance_note,
) = balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
)

print(balance_note)
print("balanced supply =", balanced_supplies.sum())
print("balanced demand =", balanced_demands.sum())
print("\nСбалансированные данные:")
print("Запасы:", balanced_supplies)
print("Спрос:", balanced_demands)

# Проверка баланса
assert np.allclose(balanced_supplies.sum(), balanced_demands.sum())

# Шаг 2: соберите LP-модель
lp_model = build_transport_lp(balanced_supplies, balanced_demands, balanced_costs)

# Вывод переменных и их стоимости
route_labels = [
    f"x_{supplier_idx + 1},{consumer_idx + 1}"
    for supplier_idx in range(len(balanced_supplier_names))
    for consumer_idx in range(len(balanced_consumer_names))
]

print("\nМатрица затрат (с фиктивным потребителем):")
balanced_cost_df = pd.DataFrame(
    balanced_costs,
    index=balanced_supplier_names,
    columns=balanced_consumer_names
)
display(balanced_cost_df)

print("\nПеременные и их стоимости:")
display(pd.DataFrame({"переменная": route_labels, "стоимость c": lp_model["c"]}))

print("\nМатрица ограничений A_eq:")
display(pd.DataFrame(lp_model["A_eq"], columns=route_labels))

print("\nПравые части ограничений b_eq:")
display(pd.DataFrame({"b_eq": lp_model["b_eq"]}))

Задача открытая: запасов больше на 15.0. Добавлен фиктивный потребитель.
balanced supply = 105.0
balanced demand = 105.0

Сбалансированные данные:
Запасы: [40. 35. 30.]
Спрос: [20. 25. 30. 15. 15.]

Матрица затрат (с фиктивным потребителем):


,Школа 1,Школа 2,Школа 3,Школа 4,Фиктивный потребитель
Комбинат A,4.0,6.0,8.0,7.0,0.0
Комбинат B,5.0,4.0,7.0,6.0,0.0
Комбинат C,6.0,5.0,4.0,8.0,0.0



Переменные и их стоимости:


,переменная,стоимость c
0,"x_1,1",4.0
1,"x_1,2",6.0
2,"x_1,3",8.0
3,"x_1,4",7.0
4,"x_1,5",0.0
5,"x_2,1",5.0
6,"x_2,2",4.0
7,"x_2,3",7.0
8,"x_2,4",6.0
9,"x_2,5",0.0



Матрица ограничений A_eq:


,"x_1,1","x_1,2","x_1,3","x_1,4","x_1,5","x_2,1","x_2,2","x_2,3","x_2,4","x_2,5","x_3,1","x_3,2","x_3,3","x_3,4","x_3,5"
0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
7,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0



Правые части ограничений b_eq:


,b_eq
0,40.0
1,35.0
2,30.0
3,20.0
4,25.0
5,30.0
6,15.0
7,15.0


In [8]:
# Решение задачи
result = linprog(
    lp_model["c"],
    A_eq=lp_model["A_eq"],
    b_eq=lp_model["b_eq"],
    bounds=lp_model["bounds"],
    method="highs",
)

print(f"Успех решения: {result.success}")
print(f"Сообщение: {result.message}")

if result.success:
    # Преобразуем решение в матрицу плана перевозок
    plan = result.x.reshape(len(balanced_supplier_names), len(balanced_consumer_names))
    plan_df = pd.DataFrame(
        plan,
        index=balanced_supplier_names,
        columns=balanced_consumer_names,
        dtype=float
    )

    print("\nОптимальный план перевозок:")
    display(plan_df.round(2))

    # Добавим итоговые строки и столбцы
    plan_with_totals = plan_df.round(2).copy()
    plan_with_totals.loc['Итого'] = plan_with_totals.sum(axis=0)
    plan_with_totals['Итого'] = plan_with_totals.sum(axis=1)

    print("\nПлан перевозок с итогами:")
    display(plan_with_totals)

    # Проверка сумм по строкам и столбцам
    print("\nПроверка ограничений:")
    print("Строки (поставщики):")
    for i, name in enumerate(balanced_supplier_names):
        row_sum = plan[i, :].sum()
        print(f"  {name}: {row_sum:.1f} = {balanced_supplies[i]:.1f} ✓")

    print("Столбцы (потребители):")
    for j, name in enumerate(balanced_consumer_names):
        col_sum = plan[:, j].sum()
        print(f"  {name}: {col_sum:.1f} = {balanced_demands[j]:.1f} ✓")

    # Общая стоимость
    total_cost = result.fun
    print(f"\nМинимальная общая стоимость перевозок: {total_cost:.2f}")

    # Проверка через assert
    assert np.allclose(plan.sum(axis=1), balanced_supplies)
    assert np.allclose(plan.sum(axis=0), balanced_demands)
    print("\nВсе проверки пройдены успешно!")

    # Анализ фиктивных маршрутов
    dummy_col_index = len(consumer_names)
    dummy_amounts = plan[:, dummy_col_index]
    print(f"\nНераспределенное питание (фиктивный потребитель):")
    for i, name in enumerate(balanced_supplier_names):
        if i < len(supplier_names):  # только реальные поставщики
            if dummy_amounts[i] > 0.01:
                print(f"  {name}: {dummy_amounts[i]:.1f} единиц останется на складе")
else:
    print("Решение не найдено!")

Успех решения: True
Сообщение: Optimization terminated successfully. (HiGHS Status 7: Optimal)

Оптимальный план перевозок:


,Школа 1,Школа 2,Школа 3,Школа 4,Фиктивный потребитель
Комбинат A,20.0,0.0,0.0,5.0,15.0
Комбинат B,0.0,25.0,0.0,10.0,0.0
Комбинат C,0.0,0.0,30.0,0.0,0.0



План перевозок с итогами:


,Школа 1,Школа 2,Школа 3,Школа 4,Фиктивный потребитель,Итого
Комбинат A,20.0,0.0,0.0,5.0,15.0,40.0
Комбинат B,0.0,25.0,0.0,10.0,0.0,35.0
Комбинат C,0.0,0.0,30.0,0.0,0.0,30.0
Итого,20.0,25.0,30.0,15.0,15.0,105.0



Проверка ограничений:
Строки (поставщики):
  Комбинат A: 40.0 = 40.0 ✓
  Комбинат B: 35.0 = 35.0 ✓
  Комбинат C: 30.0 = 30.0 ✓
Столбцы (потребители):
  Школа 1: 20.0 = 20.0 ✓
  Школа 2: 25.0 = 25.0 ✓
  Школа 3: 30.0 = 30.0 ✓
  Школа 4: 15.0 = 15.0 ✓
  Фиктивный потребитель: 15.0 = 15.0 ✓

Минимальная общая стоимость перевозок: 395.00

Все проверки пройдены успешно!

Нераспределенное питание (фиктивный потребитель):
  Комбинат A: 15.0 единиц останется на складе


In [9]:
# Анализ оптимальных маршрутов
print("\n АНАЛИЗ ОПТИМАЛЬНЫХ МАРШРУТОВ \n")

# Отделяем реальные и фиктивные маршруты
real_consumer_count = len(consumer_names)
dummy_col_index = real_consumer_count

print("Активные реальные маршруты (ненулевые перевозки):")
active_routes = []
for i, supplier in enumerate(balanced_supplier_names):
    if i >= len(supplier_names):  # пропускаем фиктивного поставщика
        continue
    for j, consumer in enumerate(balanced_consumer_names):
        if j >= real_consumer_count:  # пропускаем фиктивного потребителя
            continue
        amount = plan[i, j]
        if amount > 0.01:
            cost = costs[i, j]
            total = amount * cost
            active_routes.append({
                'Маршрут': f"{supplier} → {consumer}",
                'Объем': f"{amount:.1f}",
                'Стоимость за ед.': f"{cost:.0f}",
                'Общая стоимость': f"{total:.1f}"
            })

if active_routes:
    active_df = pd.DataFrame(active_routes)
    display(active_df)
else:
    print("Нет активных маршрутов")

print("\nНераспределенное питание (фиктивные маршруты):")
dummy_routes = []
for i, supplier in enumerate(balanced_supplier_names):
    if i >= len(supplier_names):
        continue
    amount = plan[i, dummy_col_index]
    if amount > 0.01:
        dummy_routes.append({
            'Маршрут': f"{supplier} → {DUMMY_CONSUMER_NAME}",
            'Объем': f"{amount:.1f}",
            'Причина': 'Избыток продукции'
        })

if dummy_routes:
    dummy_df = pd.DataFrame(dummy_routes)
    display(dummy_df)

# Общая стоимость по реальным маршрутам
real_total = sum(float(r['Общая стоимость']) for r in active_routes)
print(f"\nОбщая стоимость реальных перевозок: {real_total:.1f}")
print(f"Общая стоимость перевозок (включая фиктивные): {result.fun:.1f}")
print(f"Совпадение: {'✓' if abs(real_total - result.fun) < 0.01 else '✗'}")

print("\n ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТА")
print(f"Общий избыток питания составляет {dummy_amounts.sum():.1f} единиц")
print("Это означает, что часть продукции останется на складах")
print("и не будет доставлена в школы.")


 АНАЛИЗ ОПТИМАЛЬНЫХ МАРШРУТОВ 

Активные реальные маршруты (ненулевые перевозки):


,Маршрут,Объем,Стоимость за ед.,Общая стоимость
0,Комбинат A → Школа 1,20.0,4,80.0
1,Комбинат A → Школа 4,5.0,7,35.0
2,Комбинат B → Школа 2,25.0,4,100.0
3,Комбинат B → Школа 4,10.0,6,60.0
4,Комбинат C → Школа 3,30.0,4,120.0



Нераспределенное питание (фиктивные маршруты):


,Маршрут,Объем,Причина
0,Комбинат A → Фиктивный потребитель,15.0,Избыток продукции



Общая стоимость реальных перевозок: 395.0
Общая стоимость перевозок (включая фиктивные): 395.0
Совпадение: ✓

 ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТА
Общий избыток питания составляет 15.0 единиц
Это означает, что часть продукции останется на складах
и не будет доставлена в школы.


## 5. Что должно быть в отчёте

1. Таблица исходных данных и проверка баланса.
2. Полная математическая постановка через $x_{ij}$.
3. Пояснение, нужен ли фиктивный поставщик или фиктивный потребитель.
4. Векторно-матричная запись `c`, `A_eq`, `b_eq`, `bounds`.
5. Проверка через `linprog`.
6. Таблица оптимального плана перевозок.
7. Проверка сумм по строкам и столбцам.
8. Содержательная интерпретация ненулевых маршрутов.

## 6. Контрольный чек-лист

- [ ] Я сам проверил баланс до запуска solver-а.
- [ ] Я осмысленно собрал `A_eq` и `b_eq` через индекс `route_index`.
- [ ] Я проверил `bounds` и неотрицательность всех $x_{ij}$.
- [ ] Я отличаю реальные маршруты от фиктивного узла.
- [ ] Я объяснил полученный план простыми словами.
